# SMOTE Class Imbalance Comparison

This notebook evaluates the impact of **SMOTE (Synthetic Minority Over-sampling Technique)** on model performance.

We compare three scenarios:
1. **Baseline**: Original features, no SMOTE
2. **Enhanced**: Feature Engineering, no SMOTE
3. **Enhanced + SMOTE**: Feature Engineering + SMOTE

**Goal**: Improve Recall (sensitivity) for the minority class.

In [ ]:
import sys
sys.path.append('../src')

from data_preprocessing import preprocess_pipeline
from model_training import train_all_models
from evaluation import evaluate_all_models
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
RANDOM_STATE = 42
DATA_PATH = '../data/raw/hcc_dataset.csv'

## 1. Run Experiments

In [ ]:
# Scenario 1: Baseline
print("\n--- Scenario 1: Baseline ---")
data_1 = preprocess_pipeline(DATA_PATH, use_feature_engineering=False, use_smote=False, random_state=RANDOM_STATE)
models_1 = train_all_models(data_1['X_train'], data_1['y_train'], random_state=RANDOM_STATE)
res_1 = evaluate_all_models(models_1, data_1['X_test'], data_1['y_test'])
res_1['Scenario'] = 'Baseline'

# Scenario 2: Enhanced Features (No SMOTE)
print("\n--- Scenario 2: Enhanced Features ---")
data_2 = preprocess_pipeline(DATA_PATH, use_feature_engineering=True, use_smote=False, random_state=RANDOM_STATE)
models_2 = train_all_models(data_2['X_train'], data_2['y_train'], random_state=RANDOM_STATE)
res_2 = evaluate_all_models(models_2, data_2['X_test'], data_2['y_test'])
res_2['Scenario'] = 'Enhanced'

# Scenario 3: Enhanced + SMOTE
print("\n--- Scenario 3: Enhanced + SMOTE ---")
data_3 = preprocess_pipeline(DATA_PATH, use_feature_engineering=True, use_smote=True, random_state=RANDOM_STATE)
models_3 = train_all_models(data_3['X_train'], data_3['y_train'], random_state=RANDOM_STATE)
res_3 = evaluate_all_models(models_3, data_3['X_test'], data_3['y_test'])
res_3['Scenario'] = 'Enhanced + SMOTE'

## 2. Comparison Results

In [ ]:
# Combine all results
all_results = pd.concat([res_1, res_2, res_3])

# Filter for Random Forest (our best model)
rf_results = all_results[all_results['model'] == 'Random Forest']

print("\nRandom Forest Performance across Scenarios:")
print(rf_results[['Scenario', 'accuracy', 'recall', 'precision', 'f1_score']].round(4))

# Plot Recall Comparison (Focus of SMOTE)
plt.figure(figsize=(10, 6))
sns.barplot(data=all_results, x='model', y='recall', hue='Scenario', palette='rocket')
plt.title("Impact of SMOTE on Recall (Sensitivity)")
plt.ylabel("Recall Score")
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Plot F1 Comparison
plt.figure(figsize=(10, 6))
sns.barplot(data=all_results, x='model', y='f1_score', hue='Scenario', palette='viridis')
plt.title("Impact of SMOTE on F1-Score")
plt.ylabel("F1 Score")
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()